# Лабораторная работа: Кластеризация

## Цель работы

Освоить на практике основные приемы работы с самыми распространенными алгоритмами кластеризации.

## Содержание работы

1. Загрузка и подготовка датасета
2. Кластеризация методом K-средних
3. Выбор оптимального количества кластеров методом локтя
4. Нормализация признаков и повторная кластеризация
5. Кластеризация по двум признакам
6. Иерархическая кластеризация
7. Кластеризация методом DBSCAN

### 1. Загрузка и подготовка данных

Загрузим датасет с данными о клиентах торговой компании. В наборе данных представлена базовая информация: ID клиента, пол, возраст, доход и показатель потребительского поведения (Score).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Если файл mall_customers_clustering.csv доступен, загружаем его:
# df = pd.read_csv('mall_customers_clustering.csv', index_col=0)

# Для самодостаточности создадим синтетический датасет аналогичной структуры:
np.random.seed(42)
n = 200
df = pd.DataFrame({
    'CustomerID': range(1, n + 1),
    'Genre': np.random.choice(['Male', 'Female'], n),
    'Age': np.random.randint(18, 71, n),
    'Income': np.random.randint(15, 140, n) * 1000,
    'Score': np.round(np.random.rand(n), 2)
})
df.set_index('CustomerID', inplace=True)
df.head()

Визуализируем датасет в проекции на два признака (Income и Score), используя пол как цветовой признак:

In [ ]:
sns.scatterplot(x='Income', y='Score', data=df, hue='Genre')
plt.title('Распределение клиентов по доходу и потребительскому баллу')
plt.show()

Произведем минимальную техническую предобработку: удалим неинформативный атрибут CustomerID (если он есть как столбец) и преобразуем категориальный атрибут в численные признаки:

In [ ]:
# Подготовка матрицы признаков
x = df.reset_index().drop(['CustomerID'], axis=1) if 'CustomerID' in df.reset_index().columns else df.reset_index(drop=True)
# Для нашего синтетического набора df уже без CustomerID внутри, поэтому просто:
x = df.copy()
X = pd.get_dummies(x)
X.head()

### 2. Кластеризация методом K-средних

Используем самый распространенный алгоритм кластеризации — К-средних (KMeans). Разобьем выборку на 3 кластера.

In [ ]:
from sklearn.cluster import KMeans

k_means = KMeans(n_clusters=3, random_state=42).fit(X)
y_kmeans = k_means.labels_

# Визуализация кластеризации
plt.scatter(x.Income, x.Score, c=y_kmeans, s=20, cmap='viridis')
plt.title('Кластеризация K-средних (k=3)')
plt.xlabel('Income')
plt.ylabel('Score')
plt.show()

Выведем позиции центров кластеров:

In [ ]:
centers = k_means.cluster_centers_
print('Центры кластеров (в масштабированном пространстве признаков):')
print(centers)

plt.scatter(x.Income, x.Score, c=y_kmeans, s=20, cmap='viridis')
plt.scatter(centers[:, 1], centers[:, 2], c='black', s=200, alpha=0.5)
plt.title('Кластеризация K-средних с центрами кластеров (k=3)')
plt.xlabel('Income')
plt.ylabel('Score')
plt.show()

### 3. Выбор оптимального количества кластеров методом локтя

Воспользуемся метрикой WCSS (инерция) для выбора оптимального числа кластеров.

In [ ]:
wcss = []
for i in range(1, 11):
    k_means = KMeans(n_clusters=i, random_state=42)
    k_means.fit(X)
    wcss.append(k_means.inertia_)

plt.plot(range(1, 11), wcss, marker='o')
plt.xticks(range(1, 11))
plt.title('Метод локтя для выбора числа кластеров')
plt.xlabel('Количество кластеров')
plt.ylabel('WCSS (инерция)')
plt.grid(True)
plt.show()

По графику метода локтя выберем оптимальное количество кластеров и построим кластеризацию. Попробуем k=4:

In [ ]:
k_means = KMeans(n_clusters=4, random_state=42).fit(X)
y_kmeans = k_means.labels_

plt.scatter(x.Income, x.Score, c=y_kmeans, s=20, cmap='viridis')
plt.title('Кластеризация K-средних (k=4)')
plt.xlabel('Income')
plt.ylabel('Score')
plt.show()

# Попробуем k=8 для демонстрации проблемы масштаба
k_means_8 = KMeans(n_clusters=8, random_state=42).fit(X)
y_kmeans_8 = k_means_8.labels_

plt.scatter(x.Income, x.Score, c=y_kmeans_8, s=20, cmap='viridis')
plt.title('Кластеризация K-средних (k=8) — до нормализации')
plt.xlabel('Income')
plt.ylabel('Score')
plt.show()

### 4. Нормализация признаков и повторная кластеризация

Алгоритм К-средних основывается на вычислении расстояний между точками, поэтому данные необходимо нормировать.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X)
X_scaled = scaler.transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X_scaled.head()

Сравним метод локтя до и после нормализации:

In [ ]:
unscaled, scaled = [], []
for i in range(1, 11):
    unscaled.append(KMeans(n_clusters=i, random_state=42).fit(X).inertia_)
    scaled.append(KMeans(n_clusters=i, random_state=42).fit(X_scaled).inertia_)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, 11), unscaled, marker='o')
plt.xticks(range(1, 11))
plt.title('Метод локтя (исходные данные)')
plt.xlabel('Количество кластеров')
plt.ylabel('WCSS')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, 11), scaled, marker='o', color='orange')
plt.xticks(range(1, 11))
plt.title('Метод локтя (нормализованные данные)')
plt.xlabel('Количество кластеров')
plt.ylabel('WCSS')
plt.grid(True)

plt.tight_layout()
plt.show()

Построим кластеризацию на нормализованных данных:

In [ ]:
k_means = KMeans(n_clusters=4, random_state=42).fit(X_scaled)
y_kmeans_scaled = k_means.labels_

plt.scatter(x.Income, x.Score, c=y_kmeans_scaled, s=20, cmap='viridis')
plt.title('Кластеризация на нормализованных данных (k=4)')
plt.xlabel('Income')
plt.ylabel('Score')
plt.show()

### 5. Кластеризация по двум признакам

Для наглядности оставим только два признака (Income и Score) и повторим анализ.

In [ ]:
X_flat = X_scaled.drop(['Age', 'Genre_Female', 'Genre_Male'], axis=1, errors='ignore')
if 'Age' in X_scaled.columns:
    X_flat = X_scaled[['Income', 'Score']] if {'Income', 'Score'}.issubset(X_scaled.columns) else X_scaled.iloc[:, :2]
else:
    X_flat = X_scaled.iloc[:, :2]

X_flat.head()

In [ ]:
wcss_flat = []
for i in range(1, 11):
    k_means = KMeans(n_clusters=i, random_state=42)
    k_means.fit(X_flat)
    wcss_flat.append(k_means.inertia_)

plt.plot(range(1, 11), wcss_flat, marker='o')
plt.xticks(range(1, 11))
plt.title('Метод локтя для плоского датасета')
plt.xlabel('Количество кластеров')
plt.ylabel('WCSS')
plt.grid(True)
plt.show()

In [ ]:
k_means_flat = KMeans(n_clusters=5, random_state=42).fit(X_flat)
y_flat = k_means_flat.labels_
centers_flat = k_means_flat.cluster_centers_

plt.scatter(X_flat.iloc[:, 0], X_flat.iloc[:, 1], c=y_flat, s=20, cmap='viridis')
plt.scatter(centers_flat[:, 0], centers_flat[:, 1], c='black', s=200, alpha=0.5)
plt.title('Кластеризация плоского датасета (k=5)')
plt.xlabel('Income (scaled)')
plt.ylabel('Score (scaled)')
plt.show()

### 6. Иерархическая кластеризация

Построим агломеративную (иерархическую) кластеризацию и визуализируем ее в виде дендрограммы.

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram

# Обучим модель без заданного числа кластеров, чтобы построить полную дендрограмму
model = AgglomerativeClustering(distance_threshold=0, n_clusters=None)
model = model.fit(X_scaled)

# Функция для построения дендрограммы
def plot_dendrogram(model, **kwargs):
    counts = np.zeros(model.children_.shape[0])
    n_samples = len(model.labels_)
    for i, merge in enumerate(model.children_):
        current_count = 0
        for child_idx in merge:
            if child_idx < n_samples:
                current_count += 1  # leaf node
            else:
                current_count += counts[child_idx - n_samples]
        counts[i] = current_count

    linkage_matrix = np.column_stack(
        [model.children_, model.distances_, counts]
    ).astype(float)

    dendrogram(linkage_matrix, **kwargs)

plt.figure(figsize=(12, 6))
plot_dendrogram(model, truncate_mode='level', p=5)
plt.title('Дендрограмма иерархической кластеризации')
plt.xlabel('Индексы образцов (или количество образцов в узле)')
plt.ylabel('Расстояние')
plt.show()

Построим агломеративную кластеризацию с заданным числом кластеров и сравним с KMeans:

In [ ]:
agg_clustering = AgglomerativeClustering(n_clusters=5).fit(X_scaled)
y_agg = agg_clustering.labels_

plt.scatter(x.Income, x.Score, c=y_agg, s=20, cmap='viridis')
plt.title('Иерархическая кластеризация (n=5) на полном датасете')
plt.xlabel('Income')
plt.ylabel('Score')
plt.show()

# Обучим на плоских данных
agg_flat = AgglomerativeClustering(n_clusters=4).fit(X_flat)
y_agg_flat = agg_flat.labels_

plt.scatter(X_flat.iloc[:, 0], X_flat.iloc[:, 1], c=y_agg_flat, s=20, cmap='viridis')
plt.title('Иерархическая кластеризация на плоских данных (n=4)')
plt.xlabel('Income (scaled)')
plt.ylabel('Score (scaled)')
plt.show()

### 7. Кластеризация методом DBSCAN

DBSCAN оценивает плотность расположения точек в окрестностях заданного радиуса. Подберем гиперпараметры эмпирически.

In [ ]:
from sklearn.cluster import DBSCAN

# Исследуем влияние eps на количество кластеров и шума
clusters, noise = [], []
eps_values = np.logspace(-1, 1, 30)
for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=3).fit(X_flat)
    y_db = db.labels_
    clusters.append(len(set(y_db)) - (1 if -1 in y_db else 0))
    noise.append(list(y_db).count(-1))

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(eps_values, clusters, marker='o')
plt.xscale('log')
plt.title('Количество кластеров vs eps')
plt.xlabel('eps')
plt.ylabel('Количество кластеров')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(eps_values, noise, marker='o', color='red')
plt.xscale('log')
plt.title('Количество точек шума vs eps')
plt.xlabel('eps')
plt.ylabel('Точки шума')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Построим кластеризацию с подобранными параметрами
db = DBSCAN(eps=0.4, min_samples=3).fit(X_flat)
y_db = db.labels_

n_clusters_ = len(set(y_db)) - (1 if -1 in y_db else 0)
n_noise_ = list(y_db).count(-1)

print(f'Количество кластеров: {n_clusters_}')
print(f'Количество точек шума: {n_noise_}')

plt.scatter(x.Income, x.Score, c=y_db, s=20, cmap='viridis')
plt.title('Кластеризация DBSCAN')
plt.xlabel('Income')
plt.ylabel('Score')
plt.show()